In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from pprint import pprint
from imdb import IMDb
from imdb import IMDbDataAccessError
import spacy 
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import json
from IPython.display import display

In [2]:
import pickle as pkl

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
tokenizer = nltk.RegexpTokenizer(r"[^\W+\d+]+")

In [5]:
with open('datasets/stopwords.txt','r') as stopfp:
    stop_words = stopfp.readlines()

In [6]:
stop_words = stop_words[0].replace('"','').split(', ')

In [7]:
movies_df = pd.read_json('datasets/movies_df_handling.json').T

In [8]:
overview_tags = []

In [9]:
overviews = movies_df.overview.values

In [10]:
for overview in tqdm(overviews):
    tags = []
    new_overview = ' '.join(set(tokenizer.tokenize(overview.lower())))
    word_token = word_tokenize(new_overview)
    for w in word_token:
        if (w.lower() not in stop_words):
            tags.append(w)
    overview_tags.append(' '.join(tags).lower())

100%|████████████████████████████████████████████████████████████████████████████| 9704/9704 [00:05<00:00, 1665.41it/s]


In [11]:
overview_tags = pd.Series(overview_tags)

In [12]:
imdb_ids = movies_df.movie_imdb_id.values

In [13]:
tags = []

In [14]:
indeces = movies_df.index

In [15]:
indeces

Int64Index([   0,    1,    2,    3,    4,    5,    6,    7,    8,    9,
            ...
            9694, 9695, 9696, 9697, 9698, 9699, 9700, 9701, 9702, 9703],
           dtype='int64', length=9704)

In [16]:
nlp = spacy.load('en_core_web_lg')

In [17]:
genres = movies_df['genres'].apply(lambda x: ' '.join(x))

In [18]:
keywords = movies_df['keywords'].apply(lambda x: ' '.join(x))

In [ ]:
genres_doc = []
keywords_doc = []
overview_tags_doc = []

In [19]:
for i in tqdm(indeces):
    genres_doc.append(nlp(f'{genres[i]}'.strip().lower()))
    keywords_doc.append(nlp(f'{keywords[i]}'.strip().lower()))
    overview_tags_doc.append(nlp(f'{overview_tags[i]}'.strip().lower()))

100%|█████████████████████████████████████████████████████████████████████████████| 9704/9704 [01:27<00:00, 110.71it/s]


In [20]:
doc_df = pd.DataFrame()

In [21]:
movies_df.to_csv('datasets/movies_df.csv')

In [22]:
movies_df_2 = pd.read_csv('datasets/movies_df.csv')

In [23]:
movies_df['startYear'] = pd.to_datetime(movies_df['startYear']).dt.year

In [24]:
with open("movies_df.pkl",'wb') as imdb_pkl_fp:
    pkl.dump(movies_df,imdb_pkl_fp)

In [28]:
with open("movies_df.pkl",'rb') as imdb_pkl_fp:
    mv_df = pkl.load(imdb_pkl_fp)

In [50]:
def recommendation_for(imdb_id, year = 2000, freq = 10):
    # movies = movies_df.query(f'startYear >= {year}')
    title = mv_df.query(f"movie_imdb_id == 'tt{imdb_id}'").title.values[0]
    scores = []
    indices = mv_df.index
    print(f"Best recommendation for `{title}` is:")
    movie_tags = mv_df.query(f"movie_imdb_id == 'tt{imdb_id}'").tags.values[0]
    for index in indices:
        scores.append(round(movie_tags.similarity(mv_df.iloc[index].tags),2))
    recommends = pd.DataFrame({'id':mv_df.id.values,'imdb_id':mv_df.movie_imdb_id.values,'title':mv_df.title.values,'score':scores,'startYear':mv_df.startYear.values})
    recommends.drop(recommends.query('score == 1').index[0],inplace = True)
    recommends = recommends.query(f'startYear >= {year}')
    recommends = recommends.sort_values(by=['score','startYear'],ascending = False).reset_index().drop(columns = 'index')
    return recommends.head(freq)

In [51]:
movies_df.query('movie_imdb_id == "tt0241527"').tags.values[0]

adventure fantasy witch school friend friendship london, england based on novel or book magic boarding school child hero school of witchcraft chosen one school shopping fantasy world wizard christmas based on young adult novel owl powers parents deaths villain aunt blame hogwarts harness school birthday powerful wizardry harry uncovers witchcraft truth house uncle potter life place newfound waiting headmaster kindly wizard stairs learns lived

In [52]:
recommendation_for('0241527',freq = 50)

Best recommendation for `Harry Potter and the Philosopher's Stone` is:


,id,imdb_id,title,score,startYear
0,674,tt0330373,Harry Potter and the Goblet of Fire,0.94,2005
1,673,tt0304141,Harry Potter and the Prisoner of Azkaban,0.94,2004
2,531219,tt0805647,Roald Dahl's The Witches,0.93,2020
3,430447,tt6336356,Mary and The Witch's Flower,0.93,2017
4,68737,tt1121096,Seventh Son,0.93,2014
5,417384,tt3387520,Scary Stories to Tell in the Dark,0.92,2019
6,369883,tt4981636,Middle School: The Worst Years of My Life,0.92,2016
7,257445,tt1051904,Goosebumps,0.92,2015
8,12445,tt1201607,Harry Potter and the Deathly Hallows: Part 2,0.92,2011
9,73456,tt2066832,Barbie: Princess Charm School,0.92,2011
